# Step 1: Create the ML model and prepare the pipeline

This notebook creates the fraud/risk ML model **and** sets up the SEPA source table for the demo. At the end it links to the DLT pipeline (deployed via `databricks.yml`) so you can dry-run and start it.

**What this notebook does:**
1. Install dependencies, configure MLflow.
2. Generate SEPA transaction data and train a sklearn classifier.
3. Log model to MLflow and register in Unity Catalog with the **Production** alias.
4. Create the SEPA source table (`main.kyc_demo.sepa_transactions`) that the pipeline reads from and Zerobus writes into.
5. Link to the DLT pipeline and optionally start it.

**Run in Databricks Workspace** (no Databricks Connect). Use the built-in `spark` session; MLflow uses the workspace tracking server.

In [ ]:
# Install dependencies (run once, then restart Python)
%pip install mlflow[databricks] scikit-learn pandas
dbutils.library.restartPython()

After the cell above restarts Python, run the remaining cells (or **Run all**).

Load shared demo settings (catalog, schema, table names) from `config` and SEPA transaction constants.

In [ ]:
%run ../_shared/config

In [ ]:
%run ../_shared/sepa_constants

## Configuration

Set the MLflow experiment and registered model name. In Databricks, the experiment will be created under the workspace.

In [ ]:
# CATALOG, SCHEMA, CATALOG_SCHEMA come from %run ../_shared/config
# Ensure schema exists (needed for model registration in Unity Catalog)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

# MLflow experiment and model name for the demo (experiment under current user's workspace folder)
from databricks.sdk import WorkspaceClient
_current_user = WorkspaceClient().current_user.me().user_name or "unknown"
MLFLOW_EXPERIMENT_NAME = f"/Users/{_current_user}/kyc_aml_sepa_fraud_demo"
# Unity Catalog three-level model name: catalog.schema.model_name
MODEL_NAME = f"{CATALOG}.{SCHEMA}.sepa_fraud_model"
NUM_TRAINING_ROWS = 15_000  # enough for ~1% / 5% / 94% to be stable
RANDOM_STATE = 42

import mlflow
mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")

## Generate SEPA transaction data

Reuse the same company, location, and remittance lists from `sepa_constants` (loaded via `%run ../_shared/sepa_constants`). Generate a large set of rows with the same column names as the SEPA table.

In [ ]:
import random
import string
import pandas as pd
from datetime import datetime, timezone, timedelta

# COMPANIES, DEBTOR_LOCATIONS, CREDITOR_LOCATIONS, INTERMEDIARY_AGENTS,
# REMITTANCE_TYPES, CURRENCIES are loaded from sepa_constants (%run above).

random.seed(RANDOM_STATE)
base_ts = datetime.now(timezone.utc) - timedelta(days=30)
rows = []
for i in range(NUM_TRAINING_ROWS):
    debtor = random.choice(COMPANIES)
    creditor = random.choice([c for c in COMPANIES if c != debtor] or COMPANIES)
    amount = round(random.uniform(100.0, 100_000.0), 2)
    ts = base_ts + timedelta(seconds=random.randint(0, 30 * 24 * 3600))
    rows.append({
        "settlement_amount": amount,
        "currency": random.choice(CURRENCIES),
        "debtor": debtor,
        "debtor_location": random.choice(DEBTOR_LOCATIONS),
        "creditor": creditor,
        "creditor_location": random.choice(CREDITOR_LOCATIONS),
        "intermediary_agent": random.choice(INTERMEDIARY_AGENTS),
        "remittance_information": random.choice(REMITTANCE_TYPES),
        "instruction_id": "".join(random.choices(string.ascii_uppercase + string.digits, k=9)),
        "ingestion_time": ts,
    })
df = pd.DataFrame(rows)
print(f"Generated {len(df)} SEPA transaction rows.")
df.head()

## Label by risk rules and enforce target proportions

**Blocked (~1%):** High amount and (high-risk location or high-risk remittance/intermediary).  
**Medium (~5%):** One or two risk signals but not all.  
**Low (~94%):** Remainder.  

We then resample so the final dataset has exactly ~1% Blocked, ~5% Medium, ~94% Low.

In [ ]:
HIGH_RISK_LOCATIONS = {"Panama City", "George Town", "Caracas", "Lagos", "Moscow", "Minsk"}
HIGH_RISK_REMITTANCE = {"Cryptocurrency sale", "Cash transfer"}
HIGH_RISK_INTERMEDIARY = {"Offshore Trust Bank Ltd", "Atlantic Finance Ltd"}
AMOUNT_P80 = df["settlement_amount"].quantile(0.80)
AMOUNT_P60 = df["settlement_amount"].quantile(0.60)

def assign_rule_based_risk(row):
    high_amt = row["settlement_amount"] >= AMOUNT_P80
    mid_amt = row["settlement_amount"] >= AMOUNT_P60
    risky_loc = row["debtor_location"] in HIGH_RISK_LOCATIONS or row["creditor_location"] in HIGH_RISK_LOCATIONS
    risky_rem = row["remittance_information"] in HIGH_RISK_REMITTANCE
    risky_int = row["intermediary_agent"] in HIGH_RISK_INTERMEDIARY
    risk_signals = sum([risky_loc, risky_rem, risky_int])
    if high_amt and risk_signals >= 2:
        return "Blocked"
    if (high_amt and risk_signals >= 1) or (mid_amt and risk_signals >= 1) or risk_signals >= 2:
        return "Medium"
    return "Low"

df["risk_level"] = df.apply(assign_rule_based_risk, axis=1)
counts_before = df["risk_level"].value_counts()
print("Counts before resampling:", counts_before.to_dict())

# Enforce ~1% Blocked, ~5% Medium, ~94% Low
n_blocked = max(1, int(len(df) * 0.01))
n_medium = max(1, int(len(df) * 0.05))
n_low = len(df) - n_blocked - n_medium
blocked_pool = df[df["risk_level"] == "Blocked"]
medium_pool = df[df["risk_level"] == "Medium"]
low_pool = df[df["risk_level"] == "Low"]
blocked = blocked_pool.sample(n=n_blocked, replace=len(blocked_pool) < n_blocked, random_state=RANDOM_STATE) if len(blocked_pool) else blocked_pool
medium = medium_pool.sample(n=n_medium, replace=len(medium_pool) < n_medium, random_state=RANDOM_STATE + 1) if len(medium_pool) else medium_pool
low = low_pool.sample(n=n_low, replace=len(low_pool) < n_low, random_state=RANDOM_STATE + 2) if len(low_pool) else low_pool
df = pd.concat([blocked, medium, low], ignore_index=True).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
print("Counts after resampling:", df["risk_level"].value_counts().to_dict())
print("Fractions:", (df["risk_level"].value_counts(normalize=True) * 100).round(2).to_dict())

## Feature engineering and train Pipeline

Build a sklearn Pipeline: preprocessing (numeric + categorical) plus classifier. Use only columns available at inference time (no `risk_level`). Bundle the full pipeline so the demo can pass raw SEPA rows and get predictions.

In [ ]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

FEATURE_COLUMNS = [
    "settlement_amount", "currency", "debtor_location", "creditor_location",
    "remittance_information", "intermediary_agent",
]
# Add hour from ingestion_time for extra signal
df["ingestion_hour"] = pd.to_datetime(df["ingestion_time"], utc=True).dt.hour
FEATURE_COLUMNS_WITH_HOUR = FEATURE_COLUMNS + ["ingestion_hour"]

X = df[FEATURE_COLUMNS_WITH_HOUR].copy()
X["ingestion_hour"] = X["ingestion_hour"].astype(float)
# Encode target: Low=0, Medium=1, Blocked=2
RISK_ORDER = ["Low", "Medium", "Blocked"]
y = df["risk_level"].map(lambda x: RISK_ORDER.index(x) if x in RISK_ORDER else 0).values
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

numeric_features = ["settlement_amount", "ingestion_hour"]
categorical_features = ["currency", "debtor_location", "creditor_location", "remittance_information", "intermediary_agent"]
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), categorical_features),
    ],
    remainder="drop",
)
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=100, max_depth=12, random_state=RANDOM_STATE)),
])
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_val)
print("Accuracy:", accuracy_score(y_val, y_pred))
print("Macro F1:", f1_score(y_val, y_pred, average="macro"))
print(classification_report(y_val, y_pred, target_names=RISK_ORDER))

## Log to MLflow and register model

Set the experiment, log the pipeline (with signature so the demo can call with a DataFrame), log params and metrics, then register the model and add alias `production`.

In [ ]:
import mlflow
import mlflow.sklearn

mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
with mlflow.start_run():
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 12)
    mlflow.log_metric("accuracy", accuracy_score(y_val, y_pred))
    mlflow.log_metric("f1_macro", f1_score(y_val, y_pred, average="macro"))
    # Infer signature from pandas DataFrame so Spark UDF gets correct column names (numpy loses names)
    _input_sample = X_val.head(5)
    signature = mlflow.models.infer_signature(_input_sample, pipeline.predict(_input_sample))
    input_example = X_val.head(3).to_dict(orient="records")
    mlflow.sklearn.log_model(
        pipeline,
        name="model",
        signature=signature,
        input_example=input_example,
        registered_model_name=MODEL_NAME,
    )
    run_id = mlflow.active_run().info.run_id
print("Logged run_id:", run_id)

In [ ]:
# Assign the latest version to the "Production" alias (replaces deprecated stages)
client = mlflow.MlflowClient()
versions = client.search_model_versions(f"name='{MODEL_NAME}'")
latest = max(versions, key=lambda v: int(v.version))
client.set_registered_model_alias(MODEL_NAME, "Production", str(latest.version))
print(f"Model {MODEL_NAME} version {latest.version} set to Production alias.")

In [ ]:
# Load the registered model for demo scoring (Production alias)
model_uri = f"models:/{MODEL_NAME}@Production"
loaded_pipeline = mlflow.sklearn.load_model(model_uri)

# Helper: convert a raw SEPA row (e.g. from Zerobus) to the feature DataFrame for the model
def sepa_row_to_features(row):
    """row: dict or Series with settlement_amount, currency, debtor_location, creditor_location,
    remittance_information, intermediary_agent, ingestion_time."""
    if hasattr(row, "to_dict"):
        row = row.to_dict()
    ts = row.get("ingestion_time")
    hour = pd.Timestamp(ts).hour if ts is not None else 0
    return pd.DataFrame([{
        "settlement_amount": float(row["settlement_amount"]),
        "currency": row["currency"],
        "debtor_location": row["debtor_location"],
        "creditor_location": row["creditor_location"],
        "remittance_information": row["remittance_information"],
        "intermediary_agent": row["intermediary_agent"],
        "ingestion_hour": float(hour),
    }])

# Example: score a raw SEPA-style record
raw = df.drop(columns=["risk_level", "ingestion_hour"]).iloc[0].to_dict()
raw["ingestion_time"] = pd.Timestamp.now(tz="UTC")
features = sepa_row_to_features(raw)
pred = loaded_pipeline.predict(features)[0]
print("Raw SEPA row -> risk_level:", RISK_ORDER[pred])

---
## Create the SEPA source table

Create the Unity Catalog schema (if needed) and the SEPA transactions table. The DLT pipeline reads from this table; **02_zerobus_ingest** writes into it via Zerobus.

In [ ]:
# Create the SEPA source table (idempotent: no-op if table exists).
spark.sql(f"""
  CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA}.{TABLE_NAME} (
    settlement_amount DOUBLE,
    currency STRING,
    debtor STRING,
    debtor_location STRING,
    creditor STRING,
    creditor_location STRING,
    intermediary_agent STRING,
    remittance_information STRING,
    instruction_id STRING,
    ingestion_time TIMESTAMP
  )
""")
print(f"Table {CATALOG}.{SCHEMA}.{TABLE_NAME} ready.")

---
## DLT Pipeline

The DLT pipeline is deployed via `databricks.yml` (Databricks Asset Bundle). It reads from the SEPA source table created above and processes transactions through bronze, silver (with ML risk scoring), and gold layers.

Run the cell below to find the pipeline and display a direct link. **Recommended:** open the link and do a **dry run** first to verify the pipeline configuration before starting it with live data.

In [ ]:
# Find the DLT pipeline deployed by databricks.yml and display its URL.
# In dev mode the bundle prefixes the name with "[dev <username>]", so we match with 'contains'.
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
host = w.config.host.rstrip("/") if w.config and w.config.host else ""
pipeline_id = None
pipeline_actual_name = None

for p in w.pipelines.list_pipelines():
    if p.name and PIPELINE_NAME in p.name:
        pipeline_id = p.pipeline_id
        pipeline_actual_name = p.name
        break

if pipeline_id and host:
    pipeline_url = f"{host}/pipelines/{pipeline_id}"
    print(f"Pipeline: {pipeline_actual_name}")
    print(f"Pipeline ID: {pipeline_id}")
    print(f"\nOpen the pipeline in your browser:")
    print(f"  {pipeline_url}")
    print(f"\nRecommended: do a dry run first to verify the configuration.")
    displayHTML(f'<h3><a href="{pipeline_url}" target="_blank">Open Pipeline: {pipeline_actual_name}</a></h3><p>Recommended: click the link above and run a <b>dry run</b> to verify the pipeline before starting it with live data.</p>')
elif pipeline_id:
    print(f"Pipeline found: {pipeline_actual_name} (ID: {pipeline_id})")
    print("Could not determine workspace URL for direct link. Open the Pipelines UI to find it.")
else:
    print(f"Pipeline '{PIPELINE_NAME}' not found. Deploy it first with: databricks bundle deploy --target dev")

### Start the pipeline

After verifying with a dry run, run the cell below to start the pipeline. It will begin in continuous mode, waiting for data in the SEPA source table. Use **02_zerobus_ingest** to stream transactions into the table.

In [ ]:
# Start the DLT pipeline. Run this cell after verifying the pipeline with a dry run.
# Pipeline is configured with `continuous: true`, so it may already be auto-running after `bundle deploy`.
if pipeline_id:
    try:
        update = w.pipelines.start_update(pipeline_id=pipeline_id)
        print(f"Started pipeline update: {update.update_id}")
        print(f"Pipeline '{pipeline_actual_name}' is now running in continuous mode.")
    except Exception as e:
        msg = str(e)
        if "active update" in msg.lower() or "ResourceConflict" in msg:
            # Pipeline is already running an update (expected with continuous: true).
            # Extract the existing update id if present in the error message for visibility.
            import re
            m = re.search(r"'([0-9a-f-]{36})'", msg)
            existing = m.group(1) if m else "unknown"
            print(f"Pipeline '{pipeline_actual_name}' already has an active update ({existing}).")
            print("This is expected when `continuous: true` and the pipeline auto-started after deploy.")
            print("Wait for the current update to settle before running this cell again, or check the pipeline UI.")
        else:
            raise
    print("Next step: run 02_zerobus_ingest to stream SEPA transactions into the source table.")
else:
    print(f"Pipeline '{PIPELINE_NAME}' not found. Deploy it first with: databricks bundle deploy --target dev")


---
## Next step: Zerobus Ingest

The ML model is registered, the SEPA source table is ready, and the DLT pipeline is running in continuous mode waiting for data.

Open **02_zerobus_ingest** to configure Zerobus credentials and start streaming SEPA transactions into the source table. Once records arrive, the pipeline will automatically process them through bronze, silver (with ML risk scoring), and gold layers.

You can monitor the pipeline in real time from the **Pipelines** UI.